## 基本环境 · Basic setup

本章是**只读 walkthrough**，不需要填任何 TODO。它把 `feature_extraction/` 这套工程代码从 JSON 输入到模型 ready 张量的全流程拆给你看。读完你会对每个特征张量的来源、含义、形状心里有数。

This chapter is a **read-only walkthrough** — no TODOs to fill. It walks `feature_extraction/` from the input JSON down to the tensor dict the trunk consumes, so you understand where each feature comes from.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) != 'solutions':
    if os.path.isdir('solutions'):
        os.chdir('solutions')
    else:
        while os.path.basename(os.getcwd()) != 'solutions' and os.getcwd() != '/':
            os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
os.environ.setdefault('LAYERNORM_TYPE', 'torch')
print('cwd =', os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

# 第 0 章 · feature_extraction (只读)

`feature_extraction/` 把 PDB / mmCIF / FASTA / MSA / 模板 …… 杂七杂八的原始数据**翻译**成主干模型能直接吃的张量字典。它本身没有可学参数，纯数据工程，所以在 AF3 教学里我们把它当黑盒用即可。本笔记本带你走一遍数据流，让你知道:

- 输入是什么形状的 JSON
- 内部经过哪几步处理
- 输出的 `input_feature_dict` 里都有什么字段、形状、含义

## 0.1 输入 JSON

我们仓库自带的示例是 `examples/example.json`，里面是一个 7r6r 单链蛋白 + 预算好的 MSA 路径。这是用户提供 AF3 的最常见输入: 一段 `proteinChain` 加MSA / 模板等可选元数据。

In [ ]:
import json, pprint
with open('examples/example.json') as f:
    sample = json.load(f)
pprint.pprint(sample, depth=4, sort_dicts=False)

## 0.2 dataloader 入口

`get_inference_dataloader(cfg)` 是顶层入口 —— 内部读 JSON、对每个 sample 调 `SampleDictToFeatures`、MSA featurizer、template featurizer，最后吐出 PyTorch `DataLoader`。

下面用 `model/inference.py` 里相同的 config 流程构造一次:

In [ ]:
from copy import deepcopy
from configs.parser import parse_configs
from configs.configs_base import configs as base_cfg
from configs.configs_data import data_configs
from configs.configs_inference import inference_configs
from configs.configs_model_type import model_configs

MODEL_NAME = 'protenix_tiny_default_v0.5.0'

cfg = {**base_cfg, **{'data': data_configs}, **inference_configs}
cfg.update({
    'project': 'af3', 'run_name': 'walkthrough', 'base_dir': '/tmp/af3',
    'eval_interval': 0, 'log_interval': 0,
    'input_json_path': 'examples/example.json', 'model_name': MODEL_NAME,
    'triangle_attention': 'torch', 'triangle_multiplicative': 'torch',
    'enable_tf32': False, 'enable_efficient_fusion': False,
})
overrides = deepcopy(model_configs[MODEL_NAME])
def merge(d, s):
    for k, v in s.items():
        if isinstance(v, dict) and isinstance(d.get(k), dict): merge(d[k], v)
        else: d[k] = v
merge(cfg, overrides)
cfg = parse_configs(cfg, arg_str=None, fill_required_with_null=True)
cfg.model.N_cycle = 1
cfg.sample_diffusion.N_step = 5
cfg.sample_diffusion.N_sample = 1
print('config ok')

In [ ]:
from feature_extraction.inference.infer_dataloader import get_inference_dataloader

loader = get_inference_dataloader(configs=cfg)
batch = next(iter(loader))
data, atom_array, err = batch[0]
assert not err, err
print('sample:', data['sample_name'])
print('N_token =', int(data['N_token']))
print('N_atom  =', int(data['N_atom']))
print('N_msa   =', int(data['N_msa']))

## 0.3 输出特征字典

`get_inference_dataloader` 输出的 `data` 字典里只有一个 `input_feature_dict` 是真正喂给模型的张量包。下面看一下里面有什么:

In [ ]:
ifd = data['input_feature_dict']
print(f'{len(ifd)} keys total\n')
for k in sorted(ifd):
    v = ifd[k]
    import torch
    if isinstance(v, torch.Tensor):
        print(f'  {k:<32s} shape={tuple(v.shape)} dtype={str(v.dtype).replace("torch.","")}')
    elif isinstance(v, dict):
        print(f'  {k:<32s} dict, keys={list(v)[:6]}{"…" if len(v)>6 else ""}')
    else:
        print(f'  {k:<32s} {type(v).__name__}')

几条值得记住的字段:

- `restype` / `profile` / `deletion_mean` — 单序列特征 (token 级)
- `msa` / `has_deletion` / `deletion_value` — MSA 派生 (序列 + token)
- `ref_pos` / `ref_charge` / `ref_mask` / `ref_element` / `ref_atom_name_chars` —   参考几何 (原子级)
- `d_lm` / `v_lm` / `pad_info` — atom-pair 的距离与有效性 (dense-trunk 形)
- `atom_to_token_idx` / `atom_to_tokatom_idx` — 原子 → token 索引
- `asym_id` / `entity_id` / `sym_id` / `residue_index` / `token_index` — 用于 RelativePositionEncoding
- `template_*` — 模板特征 (可能没有)
- `relp` — 由 `RelativePositionEncoding.generate_relp` 在 forward 起始生成的派生特征
- `distogram_rep_atom_mask` / `token_bonds` — 后处理用

## 0.4 atom_array

另一个返回值 `atom_array` 是 biotite 的 `AtomArray`，结构数据的标准容器。推理结束写 CIF 时会用到它（把预测坐标写回 atom_array 再 dump）。

In [ ]:
print('len(atom_array) =', len(atom_array))
print('first 3 atoms:')
for atom in atom_array[:3]:
    print(f'  {atom.atom_name:>4s}  {atom.res_name:>3s} {atom.res_id:3d}  {atom.coord}')

## 章节小结

本章你没有写任何代码，目的只是知道 `feature_extraction/` 把 JSON 翻译成什么。实际推理里 `Protenix.forward` 会先调 `relative_position_encoding.generate_relp`把 `relp` 加进字典，再交给主干。

本章不挖 TODO 是有意为之: feature_extraction 是工程代码 (mmCIF 解析、MSA 处理、模板搜索等)，与 AF3 算法层关系不大，把时间花在那里收益不高。